In [1]:
from pathlib import Path
import re
import json
import numpy as np
import pandas as pd
import geopandas as gpd
from difflib import SequenceMatcher
from IPython.display import display

ROOT = Path(r"C:\Users\omrka\Documents\USB\Projects\AgriRisk and ROI Prediction\Dump")

INTEGRATED_FILE = ROOT / "data" / "processed" / "integrated" / "crop_soil_climate_integrated_2013_2025.csv"
SOIL_FILE = ROOT / "data" / "processed" / "soil" / "district_soil_texture_nrsc_5km.csv"
CLIMATE_FILE = ROOT / "data" / "processed" / "climate" / "district_climate_annual_nasa_power_2013_2025.csv"
BOUNDARY_FILE = ROOT / "data" / "raw" / "soil" / "district_boundary" / "IND_ADM2.geojson"

OUT_DIR = ROOT / "data" / "processed" / "integrated"
OUT_DIR.mkdir(parents=True, exist_ok=True)

FINAL_FILE = OUT_DIR / "crop_soil_climate_complete_2013_2025.csv"
AUDIT_FILE = OUT_DIR / "crop_soil_climate_completion_audit.csv"

for p in [INTEGRATED_FILE, SOIL_FILE, CLIMATE_FILE, BOUNDARY_FILE]:
    print(f"{p.name}: {p.exists()}")

print("Input:", INTEGRATED_FILE)
print("Output:", FINAL_FILE)


crop_soil_climate_integrated_2013_2025.csv: True
district_soil_texture_nrsc_5km.csv: True
district_climate_annual_nasa_power_2013_2025.csv: True
IND_ADM2.geojson: True
Input: C:\Users\omrka\Documents\USB\Projects\AgriRisk and ROI Prediction\Dump\data\processed\integrated\crop_soil_climate_integrated_2013_2025.csv
Output: C:\Users\omrka\Documents\USB\Projects\AgriRisk and ROI Prediction\Dump\data\processed\integrated\crop_soil_climate_complete_2013_2025.csv


In [2]:
# Load data
df = pd.read_csv(INTEGRATED_FILE)
soil = pd.read_csv(SOIL_FILE)
climate = pd.read_csv(CLIMATE_FILE)

print("Integrated:", df.shape)
print("Soil:", soil.shape)
print("Climate:", climate.shape)

print("\nIntegrated columns:")
print(df.columns.tolist())


Integrated: (67826, 30)
Soil: (735, 6)
Climate: (7800, 20)

Integrated columns:
['year', 'state', 'district', 'state_code', 'district_code', 'crop', 'season', 'area_ha', 'production_tonnes', 'yield_kg_ha', 'source', 'clayey_fraction', 'clayey_skeletal_fraction', 'loamy_fraction', 'sandy_fraction', 'soil_type', 'annual_rainfall_mm', 'annual_mean_temp_c', 'annual_max_temp_c', 'annual_min_temp_c', 'annual_relative_humidity_pct', 'annual_wind_speed_m_s', 'annual_solar_radiation', 'monsoon_rainfall_mm', 'monsoon_mean_temp_c', 'monsoon_max_temp_c', 'monsoon_min_temp_c', 'monsoon_relative_humidity_pct', 'monsoon_wind_speed_m_s', 'monsoon_solar_radiation']


In [3]:
# Standardized text keys
def clean_text(x):
    if pd.isna(x):
        return pd.NA
    s = str(x).strip().lower()
    s = re.sub(r"[\u2018\u2019\u201c\u201d]", "'", s)
    s = re.sub(r"[^a-z0-9]+", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s if s else pd.NA

for data in [df, soil, climate]:
    for col in ["state", "district"]:
        if col in data.columns:
            data[f"_{col}_key"] = data[col].map(clean_text)

# The soil file has no state column, so its district key is still useful,
# but we will resolve it against the crop-derived state mapping.
print("Text normalization complete.")


Text normalization complete.


In [4]:
# Identify environmental columns
soil_cols = [
    "clayey_fraction",
    "clayey_skeletal_fraction",
    "loamy_fraction",
    "sandy_fraction",
    "soil_type",
]

climate_cols = [
    "annual_rainfall_mm",
    "annual_mean_temp_c",
    "annual_max_temp_c",
    "annual_min_temp_c",
    "annual_relative_humidity_pct",
    "annual_wind_speed_m_s",
    "annual_solar_radiation",
    "monsoon_rainfall_mm",
    "monsoon_mean_temp_c",
    "monsoon_max_temp_c",
    "monsoon_min_temp_c",
    "monsoon_relative_humidity_pct",
    "monsoon_wind_speed_m_s",
    "monsoon_solar_radiation",
]

all_env_cols = soil_cols + climate_cols

missing_before = df[all_env_cols].isna().sum().sort_values(ascending=False)
print("Missing environmental values BEFORE recovery/imputation:")
display(missing_before.to_frame("missing_count"))
print("\nRows with any environmental missingness:", df[all_env_cols].isna().any(axis=1).sum())


Missing environmental values BEFORE recovery/imputation:


,missing_count
clayey_fraction,12130
clayey_skeletal_fraction,12130
loamy_fraction,12130
sandy_fraction,12130
soil_type,12130
annual_rainfall_mm,12130
annual_mean_temp_c,12130
annual_max_temp_c,12130
annual_min_temp_c,12130
annual_relative_humidity_pct,12130



Rows with any environmental missingness: 12130


In [5]:
#Exact key recovery

# Build crop-derived district -> unique state mapping.
# This is necessary because the supplied soil district table has no state column.
district_state_map = (
    df[["_district_key", "_state_key"]]
    .dropna()
    .drop_duplicates()
    .groupby("_district_key")["_state_key"]
    .agg(lambda s: s.iloc[0] if s.nunique() == 1 else pd.NA)
    .to_dict()
)

soil["_state_key"] = soil["_district_key"].map(district_state_map)

# Exact normalized soil lookup
soil_lookup = soil.dropna(subset=["_state_key", "_district_key"]).copy()
soil_lookup = soil_lookup.drop_duplicates(
    subset=["_state_key", "_district_key"], keep="first"
)

# Exact normalized climate lookup
climate_lookup = climate.dropna(
    subset=["_state_key", "_district_key", "year"]
).copy()
climate_lookup["year"] = pd.to_numeric(climate_lookup["year"], errors="coerce").astype("Int64")
climate_lookup = climate_lookup.dropna(subset=["year"])
climate_lookup = climate_lookup.drop_duplicates(
    subset=["_state_key", "_district_key", "year"], keep="first"
)

print("Unique exact soil keys:", len(soil_lookup))
print("Unique exact climate keys:", len(climate_lookup))


Unique exact soil keys: 616
Unique exact climate keys: 7748


In [6]:
# Recover soil values using normalized state + district keys
soil_map = soil_lookup.set_index(["_state_key", "_district_key"])[soil_cols]

df = df.merge(
    soil_map,
    how="left",
    left_on=["_state_key", "_district_key"],
    right_index=True,
    suffixes=("", "_recovered")
)

for col in soil_cols:
    recovered = f"{col}_recovered"
    if recovered in df.columns:
        df[col] = df[col].combine_first(df[recovered])
        df.drop(columns=[recovered], inplace=True)

# Recover climate values using normalized state + district + year
climate_map = climate_lookup.set_index(
    ["_state_key", "_district_key", "year"]
)[climate_cols]

df = df.merge(
    climate_map,
    how="left",
    left_on=["_state_key", "_district_key", "year"],
    right_index=True,
    suffixes=("", "_recovered")
)

for col in climate_cols:
    recovered = f"{col}_recovered"
    if recovered in df.columns:
        df[col] = df[col].combine_first(df[recovered])
        df.drop(columns=[recovered], inplace=True)

print("Exact normalized recovery completed.")
print("Rows still missing any environmental value:",
      df[all_env_cols].isna().any(axis=1).sum())


Exact normalized recovery completed.
Rows still missing any environmental value: 12062


In [7]:
# Record which rows/fields are still missing after exact recovery
for col in soil_cols + climate_cols:
    df[f"{col}_was_missing"] = df[col].isna().astype("int8")

df["soil_imputation_needed"] = df[soil_cols].isna().any(axis=1).astype("int8")
df["climate_imputation_needed"] = df[climate_cols].isna().any(axis=1).astype("int8")

print("Rows needing soil fallback:", int(df["soil_imputation_needed"].sum()))
print("Rows needing climate fallback:", int(df["climate_imputation_needed"].sum()))


Rows needing soil fallback: 11937
Rows needing climate fallback: 12062


In [8]:
# Climate fallback: state + year -> year -> global
climate_numeric = [c for c in climate_cols if c != "annual_solar_radiation" or True]

for col in climate_numeric:
    # Convert to numeric safely
    df[col] = pd.to_numeric(df[col], errors="coerce")

    state_year_median = df.groupby(
        ["_state_key", "year"], dropna=False
    )[col].transform("median")

    year_median = df.groupby("year", dropna=False)[col].transform("median")
    global_median = df[col].median()

    df[col] = df[col].fillna(state_year_median)
    df[col] = df[col].fillna(year_median)
    df[col] = df[col].fillna(global_median)

print("Climate numerical fallback completed.")


Climate numerical fallback completed.


In [9]:
# Soil numerical fallback: state -> national
soil_fraction_cols = [
    "clayey_fraction",
    "clayey_skeletal_fraction",
    "loamy_fraction",
    "sandy_fraction",
]

for col in soil_fraction_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")
    state_median = df.groupby("_state_key", dropna=False)[col].transform("median")
    df[col] = df[col].fillna(state_median)
    df[col] = df[col].fillna(df[col].median())

# Soil type fallback: state mode -> national mode
def mode_or_na(series):
    s = series.dropna().astype(str)
    if s.empty:
        return pd.NA
    return s.mode().iloc[0]

state_soil_mode = (
    df.groupby("_state_key", dropna=False)["soil_type"]
      .transform(mode_or_na)
)

df["soil_type"] = df["soil_type"].fillna(state_soil_mode)
df["soil_type"] = df["soil_type"].fillna(mode_or_na(df["soil_type"]))

print("Soil fallback completed.")


Soil fallback completed.


In [10]:
# Final completeness check
final_missing = df[all_env_cols].isna().sum().sort_values(ascending=False)

print("FINAL ENVIRONMENTAL MISSINGNESS")
display(final_missing.to_frame("missing_count"))

print("\nTotal missing environmental cells:", int(final_missing.sum()))
print("Rows with any environmental missingness:",
      int(df[all_env_cols].isna().any(axis=1).sum()))

if final_missing.sum() != 0:
    raise ValueError("Environmental missing values remain. Do not proceed.")


FINAL ENVIRONMENTAL MISSINGNESS


,missing_count
clayey_fraction,0
clayey_skeletal_fraction,0
loamy_fraction,0
sandy_fraction,0
soil_type,0
annual_rainfall_mm,0
annual_mean_temp_c,0
annual_max_temp_c,0
annual_min_temp_c,0
annual_relative_humidity_pct,0



Total missing environmental cells: 0
Rows with any environmental missingness: 0


In [11]:
# Soil fractions must be between 0 and 1
for col in soil_fraction_cols:
    df[col] = df[col].clip(0, 1)

# Soil fractions should approximately sum to 1 where all four texture classes represent the district.
texture_sum = df[soil_fraction_cols].sum(axis=1)
print("Texture fraction sum — min:", texture_sum.min())
print("Texture fraction sum — max:", texture_sum.max())

# Physical sanity checks
nonnegative_cols = [
    "annual_rainfall_mm",
    "annual_relative_humidity_pct",
    "annual_wind_speed_m_s",
    "annual_solar_radiation",
    "monsoon_rainfall_mm",
    "monsoon_relative_humidity_pct",
    "monsoon_wind_speed_m_s",
    "monsoon_solar_radiation",
]

negative_counts = {c: int((df[c] < 0).sum()) for c in nonnegative_cols}
print("\nNegative-value checks:")
print(negative_counts)

if any(v > 0 for v in negative_counts.values()):
    raise ValueError("Physically non-negative environmental variables contain negative values.")


Texture fraction sum — min: 0.0
Texture fraction sum — max: 0.9946991379310346

Negative-value checks:
{'annual_rainfall_mm': 0, 'annual_relative_humidity_pct': 0, 'annual_wind_speed_m_s': 0, 'annual_solar_radiation': 0, 'monsoon_rainfall_mm': 0, 'monsoon_relative_humidity_pct': 0, 'monsoon_wind_speed_m_s': 0, 'monsoon_solar_radiation': 0}


In [12]:
# Add compact provenance columns
df["soil_data_status"] = np.where(
    df["soil_imputation_needed"].eq(1),
    "imputed",
    "observed_or_recovered"
)

df["climate_data_status"] = np.where(
    df["climate_imputation_needed"].eq(1),
    "imputed",
    "observed_or_recovered"
)

# Overall environmental provenance
df["environmental_data_status"] = np.select(
    [
        (df["soil_imputation_needed"] == 0) & (df["climate_imputation_needed"] == 0),
        (df["soil_imputation_needed"] == 1) & (df["climate_imputation_needed"] == 0),
        (df["soil_imputation_needed"] == 0) & (df["climate_imputation_needed"] == 1),
    ],
    [
        "observed_or_recovered",
        "soil_imputed",
        "climate_imputed",
    ],
    default="soil_and_climate_imputed"
)

print(df["environmental_data_status"].value_counts(dropna=False))


environmental_data_status
observed_or_recovered       55764
soil_and_climate_imputed    11937
climate_imputed               125
Name: count, dtype: int64


In [13]:
# Drop helper normalization columns but retain provenance flags
helper_cols = [
    "_state_key",
    "_district_key",
    "soil_imputation_needed",
    "climate_imputation_needed",
]

df_final = df.drop(columns=helper_cols, errors="ignore").copy()

# Put provenance columns near the environmental fields
status_cols = [
    "soil_data_status",
    "climate_data_status",
    "environmental_data_status",
]

existing_status = [c for c in status_cols if c in df_final.columns]
base_cols = [c for c in df_final.columns if c not in existing_status]
df_final = df_final[base_cols + existing_status]

print("Final shape:", df_final.shape)


Final shape: (67826, 52)


In [14]:
# Preserve the crop master exactly in terms of row count and crop-year coverage
assert len(df_final) == 67826, f"Unexpected row count: {len(df_final)}"
assert df_final["year"].between(2013, 2024).all()

# No duplicate crop rows based on the core identifying fields
key_cols = [
    "year", "state", "district", "crop", "season",
]
dup_count = int(df_final.duplicated(subset=key_cols, keep=False).sum())
print("Duplicate core crop records:", dup_count)

# Environmental fields must be complete
assert df_final[all_env_cols].isna().sum().sum() == 0

print("\nALL COMPLETENESS CHECKS PASSED")


Duplicate core crop records: 0

ALL COMPLETENESS CHECKS PASSED


In [15]:
# Save completed dataset and audit
df_final.to_csv(FINAL_FILE, index=False)

audit_rows = []

for col in soil_cols + climate_cols:
    audit_rows.append({
        "column": col,
        "missing_before": int(df[col + "_was_missing"].sum()) if f"{col}_was_missing" in df.columns else 0,
        "missing_after": int(df_final[col].isna().sum()),
        "imputed_values": int(df_final[col].isna().sum() == 0 and df[col + "_was_missing"].sum()) if f"{col}_was_missing" in df.columns else 0
    })

audit = pd.DataFrame(audit_rows)
audit.to_csv(AUDIT_FILE, index=False)

print("Saved completed dataset:")
print(FINAL_FILE)
print("Saved audit:")
print(AUDIT_FILE)
print("\nFinal shape:", df_final.shape)
print("File size MB:", round(FINAL_FILE.stat().st_size / (1024**2), 2))


Saved completed dataset:
C:\Users\omrka\Documents\USB\Projects\AgriRisk and ROI Prediction\Dump\data\processed\integrated\crop_soil_climate_complete_2013_2025.csv
Saved audit:
C:\Users\omrka\Documents\USB\Projects\AgriRisk and ROI Prediction\Dump\data\processed\integrated\crop_soil_climate_completion_audit.csv

Final shape: (67826, 52)
File size MB: 25.72


In [16]:
# Final report
print("Rows:", len(df_final))
print("Columns:", len(df_final.columns))
print("Crop years:", sorted(df_final["year"].unique()))
print("Environmental missing cells:", int(df_final[all_env_cols].isna().sum().sum()))
print("Rows with environmental missingness:",
      int(df_final[all_env_cols].isna().any(axis=1).sum()))

print("\nEnvironmental provenance:")
print(df_final["environmental_data_status"].value_counts())

print("\nFinal file:")
print(FINAL_FILE)

if df_final[all_env_cols].isna().sum().sum() != 0:
    raise ValueError("FINAL DATASET IS NOT COMPLETE.")


Rows: 67826
Columns: 52
Crop years: [np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]
Environmental missing cells: 0
Rows with environmental missingness: 0

Environmental provenance:
environmental_data_status
observed_or_recovered       55764
soil_and_climate_imputed    11937
climate_imputed               125
Name: count, dtype: int64

Final file:
C:\Users\omrka\Documents\USB\Projects\AgriRisk and ROI Prediction\Dump\data\processed\integrated\crop_soil_climate_complete_2013_2025.csv
